In [18]:
import pandas as pd
import re
import warnings

warnings.filterwarnings('ignore')

In [19]:
df = pd.read_csv(r"E:\AI-Customer-Support-Chatbot\data\raw\raw_data.csv")
df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [20]:
nulls_count = df.isnull().sum()
print(nulls_count)

flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64


In [21]:
duplicates = df.duplicated().sum()
print(duplicates)

0


In [22]:
instruction_duplicates = df.duplicated(subset="instruction").sum()
print(instruction_duplicates)

2237


In [23]:
df = df.drop_duplicates(subset="instruction")

In [24]:
def clean_text(text: str) -> str:

    if not isinstance(text, str):
        return ''

    text = text.lower().strip()
    text = re.sub(r'\{\{.*?\}\}', '<PLACEHOLDER>', text)
    text = re.sub(r"[^\w\s<>']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['instruction_clean'] = df['instruction'].apply(clean_text)
df['response_clean']    = df['response'].apply(clean_text)

In [25]:
df.head()

,flags,instruction,category,intent,response,instruction_clean,response_clean
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...,question about cancelling order <PLACEHOLDER>,i've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...,i have a question about cancelling oorder <PLA...,i've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...,i need help cancelling puchase <PLACEHOLDER>,i can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...,i need to cancel purchase <PLACEHOLDER>,i understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...,i cannot afford this order cancel purchase <PL...,i'm sensitive to the fact that you're facing f...


In [26]:
df['category'] = df['category'].str.strip().str.upper()
df['intent']   = df['intent'].str.strip().str.lower().str.replace(' ', '_', regex=False)

In [27]:
df["tokens"] = df["instruction_clean"].apply(lambda x: x.split())
df["token_count"] = df["tokens"].apply(len)

In [28]:
df.head()

,flags,instruction,category,intent,response,instruction_clean,response_clean,tokens,token_count
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...,question about cancelling order <PLACEHOLDER>,i've understood you have a question regarding ...,"[question, about, cancelling, order, <PLACEHOL...",5
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...,i have a question about cancelling oorder <PLA...,i've been informed that you have a question ab...,"[i, have, a, question, about, cancelling, oord...",8
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...,i need help cancelling puchase <PLACEHOLDER>,i can sense that you're seeking assistance wit...,"[i, need, help, cancelling, puchase, <PLACEHOL...",6
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...,i need to cancel purchase <PLACEHOLDER>,i understood that you need assistance with can...,"[i, need, to, cancel, purchase, <PLACEHOLDER>]",6
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...,i cannot afford this order cancel purchase <PL...,i'm sensitive to the fact that you're facing f...,"[i, cannot, afford, this, order, cancel, purch...",8


In [29]:
df.to_csv(r"E:\AI-Customer-Support-Chatbot\data\processed\processed_data.csv", index=False)